In [2]:
###############################
# 1. libs
#

import pandas as pd
import numpy as np
from sklearn.metrics import precision_score
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.neural_network import MLPClassifier

import os
import pickle

In [3]:
###############################
# 2. load data
#

df = pd.read_csv('bank-full-dataset.csv', index_col = False)
df.head()

,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome,y
0,58,management,married,tertiary,no,2143,yes,no,NaN,5,may,261,1,-1,0,NaN,no
1,44,technician,single,secondary,no,29,yes,no,NaN,5,may,151,1,-1,0,NaN,no
2,33,entrepreneur,married,secondary,no,2,yes,yes,NaN,5,may,76,1,-1,0,NaN,no
3,47,blue-collar,married,NaN,no,1506,yes,no,NaN,5,may,92,1,-1,0,NaN,no
4,33,NaN,single,NaN,no,1,no,no,NaN,5,may,198,1,-1,0,NaN,no


In [4]:
###############################
# 2.1. Remove irrelevent features
#

drop_features = ['contact', 'day', 'month', 'campaign', 'pdays', 'previous', 'poutcome']
df = df.drop(drop_features, axis = 1)

df.head()

,age,job,marital,education,default,balance,housing,loan,duration,y
0,58,management,married,tertiary,no,2143,yes,no,261,no
1,44,technician,single,secondary,no,29,yes,no,151,no
2,33,entrepreneur,married,secondary,no,2,yes,yes,76,no
3,47,blue-collar,married,NaN,no,1506,yes,no,92,no
4,33,NaN,single,NaN,no,1,no,no,198,no


In [5]:
###############################
# 3.0. null count
#

count = df.isnull().sum()
display(count/df.shape[0])

age          0.000000
job          0.006370
marital      0.000000
education    0.041074
default      0.000000
balance      0.000000
housing      0.000000
loan         0.000000
duration     0.000000
y            0.000000
dtype: float64

In [6]:
###############################
# 3.1. pre-process data - cordinal encoding <=========
#

df['education'] = df['education'].fillna('unknown')
encoder = ['unknown', 'primary', 'secondary', 'tertiary']

for i, word in enumerate(encoder):
    df['education'] = df['education'].str.replace(word, str(i))
    df['education'] = df['education'].astype(np.int64, errors='ignore')

df.head(10)

,age,job,marital,education,default,balance,housing,loan,duration,y
0,58,management,married,3,no,2143,yes,no,261,no
1,44,technician,single,2,no,29,yes,no,151,no
2,33,entrepreneur,married,2,no,2,yes,yes,76,no
3,47,blue-collar,married,0,no,1506,yes,no,92,no
4,33,NaN,single,0,no,1,no,no,198,no
5,35,management,married,3,no,231,yes,no,139,no
6,28,management,single,3,no,447,yes,yes,217,no
7,42,entrepreneur,divorced,3,yes,2,yes,no,380,no
8,58,retired,married,1,no,121,yes,no,50,no
9,43,technician,single,2,no,593,yes,no,55,no


In [7]:
###############################
# 3.2. feature value count
#

cols = df.keys().tolist()

for col in cols:
    count = df[col].value_counts()
    print('----------------')
    print(count)

----------------
age
32    2085
31    1996
33    1972
34    1930
35    1894
      ... 
93       2
90       2
95       2
88       2
94       1
Name: count, Length: 77, dtype: int64
----------------
job
blue-collar      9732
management       9458
technician       7597
admin.           5171
services         4154
retired          2264
self-employed    1579
entrepreneur     1487
unemployed       1303
housemaid        1240
student           938
Name: count, dtype: int64
----------------
marital
married     27214
single      12790
divorced     5207
Name: count, dtype: int64
----------------
education
2    23202
3    13301
1     6851
0     1857
Name: count, dtype: int64
----------------
default
no     44396
yes      815
Name: count, dtype: int64
----------------
balance
 0        3514
 1         195
 2         156
 4         139
 3         134
          ... 
-381         1
 4617        1
 20584       1
 4358        1
 16353       1
Name: count, Length: 7168, dtype: int64
----------------
housi

In [52]:
###############################
# 3.3. encode qualitive features
#

enc = LabelEncoder()

cols = df.keys().tolist()

for col in cols:
    values = df[col]
    if isinstance(values[0], str):
        df[col] = enc.fit_transform(df[col].astype('str'))

df

,age,job,marital,education,default,balance,housing,loan,duration,y
0,58,4,1,3,0,2143,1,0,261,0
1,44,10,2,2,0,29,1,0,151,0
2,33,2,1,2,0,2,1,1,76,0
3,47,1,1,0,0,1506,1,0,92,0
4,33,5,2,0,0,1,0,0,198,0
...,...,...,...,...,...,...,...,...,...,...
45206,51,10,1,3,0,825,0,0,977,1
45207,71,6,0,1,0,1729,0,0,456,1
45208,72,6,1,2,0,5715,0,0,1127,1
45209,57,1,1,2,0,668,0,0,508,0


In [9]:
##############################
# 3.4. Outliers
#

cols = df.keys().tolist()

for col in cols:
    values = df[col]
    min_deviation = values.mean() - 3 * values.std()
    max_deviation = values.mean() + 3 * values.std()

    XX = df[(values >= min_deviation) & (values <= max_deviation)]
    print(df.shape[0], XX.shape[0], f'Outliers:{(1-XX.shape[0]/df.shape[0]) * 100 : .2f}%', col)

45211 44830 Outliers: 0.84% age
45211 45211 Outliers: 0.00% job
45211 45211 Outliers: 0.00% marital
45211 45211 Outliers: 0.00% education
45211 44396 Outliers: 1.80% default
45211 44466 Outliers: 1.65% balance
45211 45211 Outliers: 0.00% housing
45211 45211 Outliers: 0.00% loan
45211 44248 Outliers: 2.13% duration
45211 45211 Outliers: 0.00% y


In [10]:
###############################
# 4. X, Y
#

X = df.iloc[:, :-1]
Y = df.iloc[:, -1]

display(X.head())
display(Y.head())

,age,job,marital,education,default,balance,housing,loan,duration
0,58,4,1,3,0,2143,1,0,261
1,44,10,2,2,0,29,1,0,151
2,33,2,1,2,0,2,1,1,76
3,47,1,1,0,0,1506,1,0,92
4,33,5,2,0,0,1,0,0,198


0    0
1    0
2    0
3    0
4    0
Name: y, dtype: int32

In [11]:
###############################
# 5. train_test_split: 60/20/20
#

X = df.iloc[:, :-1]
Y = df.iloc[:, -1]

split_factor = 0.2

X_train_tmp, X_test, Y_train_tmp, Y_test = train_test_split(X, Y, test_size = split_factor, random_state = 101)

dev_split_factor = X_test.shape[0] / X_train_tmp.shape[0]

X_train, X_dev, Y_train, Y_dev = train_test_split(X_train_tmp, Y_train_tmp, test_size = dev_split_factor, random_state = 101)

print(X_train.shape, Y_train.shape, X_dev.shape, Y_dev.shape, X_test.shape, Y_test.shape)

(27125, 9) (27125,) (9043, 9) (9043,) (9043, 9) (9043,)


In [12]:
###############################
# 6.1. decision tree
#

# model_tree = DecisionTreeClassifier()
# model_tree.fit(X_train, Y_train)

In [13]:
###############################
# 6.2. Multilayer Perceptron
#

# model_anns = MLPClassifier()
# model_anns.fit(X_train, Y_train)

In [14]:
###############################
# 7. test sets
#

sets = ['Train', 'dev', 'test']
feature_sets = [X_train, X_dev, X_test]
target_sets = [Y_train, Y_dev, Y_test]

In [15]:
###############################
# 8.0. scores
#

scores = {'TREE':{},
          'MLP':{}}

In [16]:
###############################
# 8.1. recall_score decision tree --- experiment 1
#

model_tree = DecisionTreeClassifier(random_state = 2)
model_tree.fit(X_train, Y_train)

for idx, set in enumerate(sets):
    Y_pred = model_tree.predict(feature_sets[idx])
    score_precision = precision_score(target_sets[idx], Y_pred)
    scores['TREE'][set] = round(score_precision, 4)

print(scores['TREE'])

{'Train': 1.0, 'dev': 0.3322, 'test': 0.3548}


In [17]:
###############################
# 8.2. recall_score decision tree --- experiment 2
#

model_tree = DecisionTreeClassifier(random_state = 2, min_samples_leaf = 100, max_depth = 100)
model_tree.fit(X_train, Y_train)

for idx, set in enumerate(sets):
    Y_pred = model_tree.predict(feature_sets[idx])
    score_precision = precision_score(target_sets[idx], Y_pred)
    scores['TREE'][set] = round(score_precision, 4)

print(scores['TREE'])

{'Train': 0.5925, 'dev': 0.572, 'test': 0.6062}


In [18]:
############################### --- experiment 3
# 9.1 recall_score Multilayer Perceptron
#

model_anns = MLPClassifier(random_state = 2)
model_anns.fit(X_train, Y_train)

for idx, set in enumerate(sets):
    Y_pred = model_anns.predict(feature_sets[idx])
    score_precision = precision_score(target_sets[idx], Y_pred)
    scores['MLP'][set] = round(score_precision, 4)

print(scores['MLP'])

{'Train': 0.5049, 'dev': 0.5282, 'test': 0.5593}


In [19]:
############################### --- experiment 4
# 9.2 recall_score Multilayer Perceptron
#

model_anns = MLPClassifier(random_state = 2, max_iter = 1000, hidden_layer_sizes = [100, 100, 50, 25, 25], tol = 1e-4)
model_anns.fit(X_train, Y_train)

for idx, set in enumerate(sets):
    Y_pred = model_anns.predict(feature_sets[idx])
    score_precision = precision_score(target_sets[idx], Y_pred)
    scores['MLP'][set] = round(score_precision, 4)

print(scores['MLP'])

{'Train': 0.581, 'dev': 0.5541, 'test': 0.6209}


In [20]:
############################### --- experiment 5
# 9.3 recall_score Multilayer Perceptron
#

model_anns = MLPClassifier(max_iter = 800, hidden_layer_sizes = [100, 100, 100, 100, 50])
model_anns.fit(X_train, Y_train)

for idx, set in enumerate(sets):
    Y_pred = model_anns.predict(feature_sets[idx])
    score_precision = precision_score(target_sets[idx], Y_pred)
    scores['MLP'][set] = round(score_precision, 4)

print('TREE', scores['TREE'])
print(' MLP', scores['MLP'])

TREE {'Train': 0.5925, 'dev': 0.572, 'test': 0.6062}
 MLP {'Train': 0.5802, 'dev': 0.549, 'test': 0.6164}


In [21]:
print('TREE', scores['TREE'])
print(' MLP', scores['MLP'])

TREE {'Train': 0.5925, 'dev': 0.572, 'test': 0.6062}
 MLP {'Train': 0.5802, 'dev': 0.549, 'test': 0.6164}


In [22]:
###############################
# 10. Save the model
#

import os
import pickle

path = os.getcwd() + '/bank_final_model.pkl'
with open(path, 'wb') as file:
    pickle.dump(model_anns, file)

print('Complete')

Complete
